In [106]:
from dotenv import load_dotenv
import os
#from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [107]:
#import google.generativeai as genai
#from google import genai as genai2
from google.genai import types
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.tools import FunctionTool  
from google.adk.tools.agent_tool import AgentTool 
from google.adk.sessions import InMemorySessionService


In [71]:
load_dotenv(override=True)

True

In [ ]:
def send_text_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("manuagr03@gmail.com")
    to_email = To("ra255028@gmail.com")
    content = Content("text/plain", "Hello, this is a test email")
    mail = Mail(from_email, to_email, "Test Email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_text_email()

202


# Agent Workflow

In [72]:
instructions1 = "You are sales agent working for DipsAI, \
a company that provides a SaaS tool for SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are humorous, engaging sales agent working for DipsAI, \
a company that provides a SaaS tool for SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that likely to get response."

instructions3 = "You are busy sales agent working for DipsAI, \
a company that provides a SaaS tool for SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [ ]:
sales_agent1 = Agent(
        name="Professional_Sales_Agent",
        instruction=instructions1,
        model="gemini-2.5-flash"
)

In [42]:
print(sales_agent1)

name='Professional_Sales_Agent' description='Write a subject for a cold sales email' parent_agent=None sub_agents=[] before_agent_callback=None after_agent_callback=None model='gemini-2.0-flash-lite' instruction='You are sales agent working for DipsAI, a company that provides a SaaS tool for SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.' global_instruction='' static_instruction=None tools=[] generate_content_config=None disallow_transfer_to_parent=False disallow_transfer_to_peers=False include_contents='default' input_schema=None output_schema=None output_key=None planner=None code_executor=None before_model_callback=None after_model_callback=None on_model_error_callback=None before_tool_callback=None after_tool_callback=None on_tool_error_callback=None


In [66]:
session_memory = InMemorySessionService()

session = await session_memory.create_session(
        app_name="sales",
        user_id="user_1"
    )

result = Runner(
    agent=sales_agent1,
    app_name="sales",
    session_service=session_memory
)


async for event in result.run_async(
        user_id="user_1",
        session_id=session.id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text="Write a cold sales email")]
        )
    ):
        try:
            if event.is_final_response():
                print(event.content.parts[0].text)
        except Exception as e:
            print("Rate limited, waiting 45s...")
            await asyncio.sleep(45)



Subject: Streamline Your SOC 2 Compliance with AI

Dear [Recipient Name],

In today's fast-paced digital environment, achieving and maintaining SOC 2 compliance can often feel like a perpetual, resource-intensive challenge, diverting critical talent from strategic initiatives. The manual processes involved in evidence collection, continuous monitoring, and audit preparation frequently lead to inefficiencies and unexpected complexities.

At DipsAI, we specialize in transforming this burden. Our AI-powered SaaS platform is engineered to automate and accelerate your entire SOC 2 compliance journey. DipsAI intelligently collects necessary evidence, provides real-time insights into your compliance posture, and generates comprehensive, audit-ready reports, significantly reducing the manual effort, time commitment, and potential for human error. This means your team can achieve continuous compliance and be audit-ready with unprecedented efficiency.

We believe DipsAI offers a smarter, more st

In [102]:
sales_agent1 = Agent(
        name="Professional_Sales_Agent",
        instruction=instructions1,
        model="gemini-2.0-flash"
)

sales_agent2 = Agent(
        name="Engaging_Sales_Agent",
        instruction=instructions2,
        model="gemini-2.0-flash"
)

sales_agent3 = Agent(
        name="Busy_Sales_Agent",
        instruction=instructions3,
        model="gemini-2.0-flash"
)

In [108]:
agent1_tool = AgentTool(agent=sales_agent1)
agent2_tool = AgentTool(agent=sales_agent2)
agent3_tool = AgentTool(agent=sales_agent3)

In [90]:

def send_email(body: str) -> dict:
    """Send out an email with the given body to all sales prospects."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("manuagr03@gmail.com")
    to_email = To("ra255028@gmail.com")
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Test Email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    #print(response.status_code)
    return {"status": "success"}


In [109]:
email_tool  = FunctionTool(func=send_email)

In [91]:
instructions = """You are an sales Manager at DipsAI. 
Your goal is to find the single best email by using all three sales_agent and sub-agents to generate three different email drafts.
Follow these steps carefully:
1: Generate Drafts: Use all three sales_agent and sub-agents to generate three different email drafts, do not proceed untill all three drafts are ready.
2: Evaluate and Select: Review the draft and select single best email using your judgement of which one is most effective.
3: Use the send_email tool to send the best email (and only the best email) to the user.

Rules:
- Never write drafts yourself — always use the sub-agents.
- Hand off exactly ONE email to the Email_Manager.

"""

In [110]:
sales_manager = Agent(
        name="sales_manager",
        instruction=instructions,
        model="gemini-2.0-flash",
        tools = [agent1_tool, agent2_tool, agent3_tool, email_tool]
)

In [111]:
message = "Send a cold sales email addressed to 'Dear CEO'"
APP_NAME = "sales_app" 

session_memory1 = InMemorySessionService()

session_m = await session_memory1.create_session(
        app_name=APP_NAME,
        user_id="user_2"
    )


result = Runner(
    agent=sales_manager,
    app_name=APP_NAME,
    session_service=session_memory1
)

async for event in result.run_async(
        user_id="user_2",
        session_id=session_m.id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=message)]
        )):
        try:
            if event.is_final_response():
                print("Done:", event.content.parts[0].text)
        except Exception as e:
            print("Rate limited, waiting 45s...")
            await asyncio.sleep(45)

_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 41.479977146s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help'

In [ ]:
from dotenv import load_dotenv
import os
import asyncio
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
from google.genai import types
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import FunctionTool
from google.adk.tools.agent_tool import AgentTool

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "0"
load_dotenv(override=True)

# ─────────────────────────────────────────────────────────────────
# 1. USE gemini-2.0-flash-lite  ← Free tier friendlier model
#    Flash-lite has higher free RPM than flash
# ─────────────────────────────────────────────────────────────────
MODEL = "gemini-2.5-flash-lite"

# ─────────────────────────────────────────────────────────────────
# 2. RETRY HELPER — exponential backoff on 429s
#    Free tier limit: 15 RPM, 1M TPM, 1500 req/day (flash-lite)
#    This prevents crashes and auto-recovers from quota hits
# ─────────────────────────────────────────────────────────────────
async def with_retry(coro_fn, max_retries=5, base_delay=30):
    """
    Retry an async coroutine with exponential backoff.
    Handles 429 Resource Exhausted from Gemini free quota.
    """
    for attempt in range(max_retries):
        try:
            return await coro_fn()
        except Exception as e:
            err = str(e).lower()
            is_quota = "429" in err or "resource exhausted" in err or "quota" in err
            if is_quota and attempt < max_retries - 1:
                wait = base_delay * (2 ** attempt)  # 30s, 60s, 120s, 240s...
                print(f"⚠️  Quota hit (attempt {attempt+1}/{max_retries}). "
                      f"Retrying in {wait}s...")
                await asyncio.sleep(wait)
            else:
                raise  # re-raise if not quota or out of retries

# ─────────────────────────────────────────────────────────────────
# 3. THROTTLED RUNNER HELPER
#    Adds a mandatory delay between each agent call
#    to stay under 15 RPM (1 call per 4s minimum)
# ─────────────────────────────────────────────────────────────────
INTER_CALL_DELAY = 5  # seconds between agent invocations

async def run_agent_with_throttle(
    runner: Runner,
    user_id: str,
    session_id: str,
    message: str,
    delay_before: float = 0.0
) -> str:
    """Run a single agent call with optional pre-delay and retry logic."""

    if delay_before > 0:
        print(f"⏳ Throttling: waiting {delay_before}s before next call...")
        await asyncio.sleep(delay_before)

    collected = []

    async def _run():
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=types.Content(
                role="user",
                parts=[types.Part(text=message)]
            )
        ):
            if event.is_final_response():
                collected.append(event.content.parts[0].text)

    await with_retry(_run)
    return collected[0] if collected else ""


# ─────────────────────────────────────────────────────────────────
# 4. SUB-AGENT DEFINITIONS
#    Kept instructions SHORT → fewer tokens → less quota burn
# ─────────────────────────────────────────────────────────────────
instructions1 = (
    "You are a professional sales agent at DipsAI, which offers an AI-powered "
    "SaaS tool for SOC2 compliance and audit preparation. "
    "Write one professional, formal cold email. Be concise — max 100 words."  # ← token limit hint
)

instructions2 = (
    "You are a witty sales agent at DipsAI, which offers an AI-powered "
    "SaaS tool for SOC2 compliance and audit preparation. "
    "Write one humorous, engaging cold email. Be concise — max 100 words."
)

instructions3 = (
    "You are a direct sales agent at DipsAI, which offers an AI-powered "
    "SaaS tool for SOC2 compliance and audit preparation. "
    "Write one ultra-concise cold email. Max 60 words — no fluff."
)

sales_agent1 = Agent(
    name="Professional_Sales_Agent",
    instruction=instructions1,
    model=MODEL   # ✅ flash-lite for all sub-agents
)

sales_agent2 = Agent(
    name="Engaging_Sales_Agent",
    instruction=instructions2,
    model=MODEL
)

sales_agent3 = Agent(
    name="Concise_Sales_Agent",
    instruction=instructions3,
    model=MODEL
)

# ─────────────────────────────────────────────────────────────────
# 5. SEND EMAIL FUNCTION TOOL
# ─────────────────────────────────────────────────────────────────
def send_email(body: str) -> dict:
    """
    Send a cold sales email to prospects.

    Args:
        body: The full email body to send.

    Returns:
        A dict with 'status' key.
    """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("xyz@gmail.com")
    to_email   = To("yzq@gmail.com")
    content    = Content("text/plain", body)
    mail       = Mail(from_email, to_email, "Cold Email from DipsAI", content).get()
    response   = sg.client.mail.send.post(request_body=mail)
    print(f"📧 SendGrid status: {response.status_code}")
    return {"status": "success"}

# ─────────────────────────────────────────────────────────────────
# 6. SALES MANAGER
#    tools[] wired up correctly with AgentTool + FunctionTool
# ─────────────────────────────────────────────────────────────────
agent1_tool = AgentTool(agent=sales_agent1)
agent2_tool = AgentTool(agent=sales_agent2)
agent3_tool = AgentTool(agent=sales_agent3)
email_tool  = FunctionTool(func=send_email)

manager_instructions = """
You are the Sales Manager at DipsAI.

Steps (follow exactly):
1. Call Professional_Sales_Agent with the user brief → get draft 1.
2. Call Engaging_Sales_Agent with the user brief → get draft 2.
3. Call Concise_Sales_Agent with the user brief → get draft 3.
4. Pick the single best draft. State which one and why in one sentence.
5. Call send_email ONCE with the chosen draft body.

Rules:
- Never write drafts yourself.
- Call send_email exactly once.
- Keep your own responses SHORT — you are an orchestrator, not a writer.
"""

# ─────────────────────────────────────────────────────────────────
# 7. OPTION A: Let sales_manager orchestrate via tools (ADK native)
#    Use this if you want full ADK agent-as-tool orchestration.
#    The manager calls sub-agents sequentially by itself.
# ─────────────────────────────────────────────────────────────────
sales_manager = Agent(
    name="sales_manager",
    instruction=manager_instructions,
    model=MODEL,
    tools=[agent1_tool, agent2_tool, agent3_tool, email_tool]
)

# ─────────────────────────────────────────────────────────────────
# 8. OPTION B: Manual sequential orchestration (MORE quota-safe)
#    You control the delays explicitly between each sub-agent call.
#    Uncomment this block and comment out the runner.run_async
#    in main() if Option A keeps hitting quota.
# ─────────────────────────────────────────────────────────────────
async def manual_orchestration(session_service, user_id, session_id, brief):
    """
    Manually call sub-agents one-by-one with throttling.
    More quota-safe than letting the manager fire all three rapidly.
    """
    runners = {
        "professional": Runner(agent=sales_agent1, app_name=APP_NAME, session_service=session_service),
        "engaging"     : Runner(agent=sales_agent2, app_name=APP_NAME, session_service=session_service),
        "concise"      : Runner(agent=sales_agent3, app_name=APP_NAME, session_service=session_service),
    }

    drafts = {}
    delay  = 0.0

    for name, runner in runners.items():
        print(f"\n🤖 Calling {name} agent...")
        draft = await run_agent_with_throttle(
            runner, user_id, session_id, brief, delay_before=delay
        )
        drafts[name] = draft
        print(f"✅ {name} draft received ({len(draft)} chars)")
        delay = INTER_CALL_DELAY  # apply delay for all calls after first

    return drafts


# ─────────────────────────────────────────────────────────────────
# 9. MAIN
# ─────────────────────────────────────────────────────────────────
APP_NAME = "sales_app"
USER_ID  = "user_2"

async def main():
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID
    )

    brief = "Write a cold sales email addressed to 'Dear CEO'"

    # ── OPTION A: ADK native orchestration via sales_manager ──────
    print("\n🚀 Starting sales_manager orchestration...\n")
    manager_runner = Runner(
        agent=sales_manager,
        app_name=APP_NAME,
        session_service=session_service
    )

    result = await run_agent_with_throttle(
        runner=manager_runner,
        user_id=USER_ID,
        session_id=session.id,
        message=brief,
        delay_before=0.0
    )
    print(f"\n✅ Final manager output:\n{result}")

    # ── OPTION B: Manual sequential (uncomment if quota issues persist) ──
    # print("\n🚀 Running manual sequential orchestration...\n")
    # drafts = await manual_orchestration(
    #     session_service, USER_ID, session.id, brief
    # )
    # # Let manager pick the best — pass all drafts to it
    # combined = "\n\n---\n\n".join(
    #     f"[{k.upper()} DRAFT]\n{v}" for k, v in drafts.items()
    # )
    # pick_prompt = (
    #     f"Here are 3 email drafts:\n\n{combined}\n\n"
    #     f"Pick the best one and call send_email with its body."
    # )
    # manager_runner = Runner(
    #     agent=sales_manager,
    #     app_name=APP_NAME,
    #     session_service=session_service
    # )
    # result = await run_agent_with_throttle(
    #     manager_runner, USER_ID, session.id, pick_prompt
    # )
    # print(f"\n✅ Manager picked and sent:\n{result}")


if __name__ == "__main__":
    # asyncio.run(main())
    await main()

<Task pending name='Task-206' coro=<main() running at /var/folders/7g/5kslqxp57m331gyj9hnd4t580000gn/T/ipykernel_55604/1046884169.py:221>>


🚀 Starting sales_manager orchestration...



In [120]:
# ── CELL 5: Minimal ADK call with explicit key via google.genai client ──
import os
from google import genai
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# ADK uses google.genai under the hood — configure it explicitly
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

os.environ["GOOGLE_API_KEY"]   = os.environ.get("GOOGLE_API_KEY", "")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "0"  # ← force AI Studio, not Vertex AI

try:
    agent = Agent(
        name="test_agent",
        instruction="Reply with exactly: ADK is working",
        model="gemini-2.5-flash-lite"
    )
    session_service = InMemorySessionService()

    async def test_adk():
        session = await session_service.create_session(
            app_name="test", user_id="u1"
        )
        runner = Runner(
            agent=agent,
            app_name="test",
            session_service=session_service
        )
        async for event in runner.run_async(
            user_id="u1",
            session_id=session.id,
            new_message=types.Content(
                role="user",
                parts=[types.Part(text="hello")]
            )
        ):
            if event.is_final_response():
                print(f"✅ ADK works: {event.content.parts[0].text}")

    await test_adk()

except Exception as e:
    print(f"❌ ADK failed: {type(e).__name__}: {e}")

✅ ADK works: ADK is working


# Handling Rate limit and Quota Managemnt errors

In [15]:
from collections import deque
import time

In [16]:
class RateLimiter:
    """ Token bucket rate limit error for google gemini API """
    def __init__(self, requests_per_minute: int = 15):
        self.rpm = requests_per_minute
        self.requests = deque()

    async def acquire(self):
        now = time.time()
        """ Acquire a token from the bucket """
        # Remove requests older than 60 seconds
        while self.requests and self.requests[0]< now - 60:
            self.requests.popleft()

        if len(self.requests) >= self.rpm:
            wait_time = 60 - (now - self.requests[0])
            print(f"Rate limit reached. Waiting {wait_time: .1f}s...")
            await asyncio.sleep(wait_time)

        self.requests.append(time.time())

rate_limiter = RateLimiter(requests_per_minute=15)
    
    

# Building a Reusable GeminiAgent Class


In [18]:
import json
import logging
from typing import Any, Callable, Optional
from dataclasses import dataclass, field

In [19]:
logger = logging.getLogger(__name__)

In [20]:
@dataclass
class GeminiAgent:
    name: str
    instructions: str   # system instruction
    tools: list = field(default_factory=list)
    handoffs: list = field(default_factory=list)    # other gemini agent instances
    model_name: str = "gemini-2.0-flash"
    temperature: str = 0.7
    max_output_tokens: int = 8192
    handoffs_description: str = ""

    def __post_init__(self):
        self._tool_functions: dict[str, Callable] = {}
        self._model = self._build_model()

    def _build_model(self) -> genai.GenerativeModel:
        all_gemini_tools = []

        
        # Register function tools
        for tool in self.tools:
            if hasattr(tool, '_gemini_declaration'):
                all_gemini_tools.append(tool._gemini_declaration)
                self._tool_functions[tool._gemini_declaration.name] = tool
            elif isinstance(tool, GeminiAgent):
                decl = self._agent_as_function_decl(tool)
                all_gemini_tools.append(decl)
                self._tool_functions[tool.name.replace(" ", "-").lower()] = tool

        # Register handoffs as callable tools
        for handoff_agent in self.handoffs:
            decl = self._agent_as_function_decl(handoff_agent, is_handoff=True)
            all_gemini_tools.append(decl)
            fn_name = handoff_agent.name.replace(" ", "-").lower()
            self._tool_functions[fn_name] = handoff_agent
